# Process Dependency Graph — Visualization
Visualizes the sensor correlation graph used as input to the GNN.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

FIG_DIR = Path('../paper/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load graph
graph_data = torch.load('../data/processed/graph.pt')
edge_index = graph_data['edge_index'].numpy()
edge_weight = graph_data['edge_weight'].numpy()
feature_names = graph_data['feature_names']
n_nodes = graph_data['num_nodes']
n_edges = graph_data['num_edges']

print(f'Nodes: {n_nodes}, Edges: {n_edges}')

In [ ]:
# Build NetworkX graph
G = nx.Graph()
G.add_nodes_from(range(n_nodes))
for i in range(n_edges):
    src, dst = edge_index[0][i], edge_index[1][i]
    G.add_edge(src, dst, weight=float(edge_weight[i]))

# Compute degree for node sizing
degrees = dict(G.degree())
node_sizes = [degrees[n] * 20 + 50 for n in G.nodes()]
node_colors = [degrees[n] for n in G.nodes()]

print(f'Connected components: {nx.number_connected_components(G)}')
print(f'Average clustering coefficient: {nx.average_clustering(G):.4f}')

In [ ]:
# Visualize subgraph — top 50 highest degree nodes for clarity
top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:50]
subgraph = G.subgraph(top_nodes)

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(subgraph, seed=42, k=0.5)
sub_degrees = dict(subgraph.degree())
node_colors = [sub_degrees[n] for n in subgraph.nodes()]
node_sizes = [sub_degrees[n] * 30 + 100 for n in subgraph.nodes()]

nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors,
                       node_size=node_sizes, cmap=plt.cm.plasma,
                       alpha=0.9, ax=ax)
nx.draw_networkx_edges(subgraph, pos, alpha=0.3, edge_color='gray', ax=ax)
nx.draw_networkx_labels(subgraph, pos,
                        labels={n: feature_names[n].replace('sensor_', 'S') for n in subgraph.nodes()},
                        font_size=6, ax=ax)

ax.set_title('Sensor Process Dependency Graph\n(Top 50 nodes by degree)',
             fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'process_graph.png', bbox_inches='tight', dpi=150)
plt.show()
print('Graph visualization saved.')

In [ ]:
# Degree distribution
degree_vals = list(degrees.values())
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(degree_vals, bins=30, color='#3498db', edgecolor='black', linewidth=0.8)
ax.set_xlabel('Node Degree', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Degree Distribution of Sensor Graph', fontsize=13, fontweight='bold')
ax.axvline(np.mean(degree_vals), color='red', linestyle='--',
           label=f'Mean degree: {np.mean(degree_vals):.1f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'degree_distribution.png', bbox_inches='tight')
plt.show()